# 03 — Engineer Baseline xG Features
Re-read raw event JSONs to extract pre-shot body-part tags, merge onto the with_matches
parquet, and compute geometry features (distance to goal, shot angle). Saves the baseline
xG feature table used for modelling.

The canonical shot-attempt dataset (from notebook 01) includes standard shots, in-game
penalties, and direct free-kick shots. `is_penalty` and `is_direct_free_kick` are passed
through from upstream — not re-derived here.

**Locked baseline model inputs:** `distance_to_goal`, `shot_angle_rad`, `is_penalty`,
`is_direct_free_kick`, `is_left_foot`, `is_right_foot`, `is_header` — target: `is_goal`.

**Feature contract:**
- Use `is_left_foot`, `is_right_foot`, `is_header` as binary inputs — do NOT additionally encode `shot_body_part` (avoids double-encoding).
- If all three body-part booleans are used together in a linear model, drop one reference category to avoid dummy-variable redundancy.
- `shot_angle_deg` is for inspection/reporting only — use `shot_angle_rad` as the model input.
- `x`, `y`, and provenance columns (`winner`, `match_status`, etc.) are retained for auditability but are not model inputs.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from src.config import DATA_DIR, EVENT_DIR, ALL_LEAGUES
from src.data_loader import load_events, load_parquet
from src.feature_engineering import build_tag_rows, add_body_part_label, add_shot_geometry

## Step 0: Path validation

In [ ]:
assert (DATA_DIR / "wyscout_shots_with_matches.parquet").exists(), "Run 02 first"
assert EVENT_DIR.exists(), f"Missing: {EVENT_DIR}"
print("Paths OK")

## Step 1: Load with_matches shots

In [ ]:
shots = load_parquet(DATA_DIR / "wyscout_shots_with_matches.parquet")
assert "tags" not in shots.columns, "Unexpected tags column — notebook 02 should be scalar-only"

# New upstream columns must be present
for col in ["eventName", "subEventName", "is_penalty", "is_direct_free_kick"]:
    assert col in shots.columns, f"Missing column from upstream: {col}"

# Normalise flag dtypes
shots["is_penalty"]          = shots["is_penalty"].astype("int8")
shots["is_direct_free_kick"] = shots["is_direct_free_kick"].astype("int8")

# Upstream flag validity
assert shots["is_penalty"].isin([0, 1]).all(), "is_penalty not binary"
assert shots["is_direct_free_kick"].isin([0, 1]).all(), "is_direct_free_kick not binary"
assert not ((shots["is_penalty"] == 1) & (shots["is_direct_free_kick"] == 1)).any(), \
    "Mutual exclusivity violated: a row has both flags = 1"

# Event/subevent pair consistency (carried forward from notebook 02)
assert (shots.loc[shots["is_penalty"] == 1, "eventName"] == "Free Kick").all()
assert (shots.loc[shots["is_penalty"] == 1, "subEventName"] == "Penalty").all()
assert (shots.loc[shots["is_direct_free_kick"] == 1, "eventName"] == "Free Kick").all()
assert (shots.loc[shots["is_direct_free_kick"] == 1, "subEventName"] == "Free kick shot").all()
assert (shots.loc[
    (shots["is_penalty"] == 0) & (shots["is_direct_free_kick"] == 0), "eventName"
] == "Shot").all()

print(f"Loaded: {len(shots):,} attempts, {shots.shape[1]} columns")
print(f"  Penalties:        {(shots['is_penalty'] == 1).sum():,}")
print(f"  Direct FK shots:  {(shots['is_direct_free_kick'] == 1).sum():,}")

## Step 2: Re-read raw events to extract body-part tags

Only pre-shot tags are used. Tags encoding post-shot outcomes are excluded:

| Tag ID | Meaning | Used? |
|--------|---------|-------|
| 401 | Left foot | ✅ |
| 402 | Right foot | ✅ |
| 403 | Head | ✅ |
| 201 | On target | ❌ post-shot |
| 2101 | Blocked | ❌ post-shot |
| 1201–1223 | Goal target zones | ❌ post-shot |
| 1801 | Accurate | ❌ post-shot |
| 1802 | Not accurate | ❌ post-shot |

In [ ]:
# verbose=True preserves the current per-league console output exactly.
tags_df = build_tag_rows(ALL_LEAGUES, load_events, verbose=True)

## Step 3: Validate body-part coverage before merge

In [5]:
bp_sum = tags_df[["is_left_foot","is_right_foot","is_header"]].sum(axis=1)
n_zero  = (bp_sum == 0).sum()
n_multi = (bp_sum > 1).sum()

print(f"Attempts with no body-part tag:       {n_zero:,}")
print(f"Attempts with multiple body-part tags: {n_multi:,}")

if n_zero > 0 or n_multi > 0:
    print("WARNING: body part is not perfectly exclusive/exhaustive — unknown category will be used")
else:
    print("Body-part tags are mutually exclusive and exhaustive — hard assert safe")
    assert (bp_sum == 1).all()

Attempts with no body-part tag:       0
Attempts with multiple body-part tags: 0
Body-part tags are mutually exclusive and exhaustive — hard assert safe


## Step 4: Build shot_body_part (vectorised)

In [ ]:
tags_df = add_body_part_label(tags_df)

## Step 5: Merge body-part flags onto shots

In [7]:
# Shot identity definition inherited from notebooks 01/02 — not a new key contract
MERGE_KEYS = ["league","matchId","playerId","teamId","eventSec","matchPeriod"]

# Guard: null or duplicate merge keys would cause silent join failures
assert shots[MERGE_KEYS].notna().all().all(),   "Null merge keys in shots"
assert tags_df[MERGE_KEYS].notna().all().all(), "Null merge keys in tags_df"
assert not shots[MERGE_KEYS].duplicated().any(),   "Duplicate shot keys in shots — validate='one_to_one' would fail"
assert not tags_df[MERGE_KEYS].duplicated().any(), "Duplicate shot keys in tags_df — validate='one_to_one' would fail"

# Hard pipeline check: counts must match exactly; any mismatch = upstream filtering drift
if len(tags_df) != len(shots):
    diag = (
        tags_df.groupby("league").size().rename("tags_df")
        .to_frame()
        .join(shots.groupby("league").size().rename("shots"))
    )
    diag["diff"] = diag["tags_df"] - diag["shots"]
    print(diag)
    raise AssertionError(
        f"tags_df row count ({len(tags_df)}) \u2260 shots row count ({len(shots)}) "
        "\u2014 see per-league diagnostic above"
    )

pre_merge_n = len(shots)
shots = shots.merge(
    tags_df[MERGE_KEYS + ["is_left_foot","is_right_foot","is_header","shot_body_part"]],
    on=MERGE_KEYS,
    how="left",
    validate="one_to_one",
)
assert len(shots) == pre_merge_n, f"Row count changed after merge: {pre_merge_n} \u2192 {len(shots)}"
n_unmatched = shots["shot_body_part"].isna().sum()
assert n_unmatched == 0, f"{n_unmatched} shots have null body_part after merge \u2014 check join keys"
print(f"Merged: {len(shots):,} shots")
print(shots["shot_body_part"].value_counts())

Merged: 43,040 shots
shot_body_part
right_foot    22289
left_foot     14230
head           6521
Name: count, dtype: int64


## Step 6: Geometry features

> **Dataset assumption:** Wyscout shot event coordinates are already oriented so that the
> attacking goal is at x = 100. All shot events attack in the same direction — no mirroring required.

Shot angle is computed via the law of cosines (robust for wide / behind-post positions).

In [ ]:
shots = add_shot_geometry(shots)

## Step 7: Assertions

In [9]:
# Body part — no row may have more than one body-part flag set
bp_sum = shots[["is_left_foot","is_right_foot","is_header"]].sum(axis=1)
assert (bp_sum <= 1).all(), "Multiple body-part tags on same attempt after merge"
assert shots["shot_body_part"].notna().all(), "Null shot_body_part"

# Unknown body-part check
n_unknown = (shots["shot_body_part"] == "unknown").sum()
print(f"Unknown body part: {n_unknown:,}")
if n_unknown > 0:
    unk = shots[shots["shot_body_part"] == "unknown"]
    assert (unk[["is_left_foot", "is_right_foot", "is_header"]] == 0).all().all(), \
        "Unknown body-part rows must have all three body-part booleans = 0"

# Geometry
assert shots["distance_to_goal"].ge(0).all(), "Negative distance"
assert shots["distance_to_goal"].le(120).all(), "Distance implausibly large"
assert shots["shot_angle_rad"].between(0, np.pi).all(), "Angle out of [0, π]"
assert shots["shot_angle_deg"].between(0, 180).all(), "Angle out of [0°, 180°]"

print("All assertions passed")
print(shots[["distance_to_goal","shot_angle_deg"]].describe().round(2))

Unknown body part: 0
All assertions passed
       distance_to_goal  shot_angle_deg
count          43040.00        43040.00
mean              18.91           23.46
std                8.52           14.40
min                0.68            0.00
25%               12.42           14.16
50%               17.57           18.43
75%               25.43           28.76
max              103.95          180.00


## Step 8: Summary display

In [10]:
display(shots.groupby("league").agg(
    n_shots=("is_goal","count"),
    goal_rate=("is_goal","mean"),
    header_rate=("is_header","mean"),
    mean_distance=("distance_to_goal","mean"),
    mean_angle_deg=("shot_angle_deg","mean"),
).round(3))

,n_shots,goal_rate,header_rate,mean_distance,mean_angle_deg
league,,,,,
England,8881,0.111,0.148,18.595,23.999
France,8977,0.111,0.148,19.417,23.108
Germany,7290,0.114,0.156,18.579,23.871
Italy,9347,0.105,0.146,19.223,22.901
Spain,8545,0.116,0.160,18.639,23.541


## Step 9: Save

In [11]:
# ── Baseline xG model feature columns ──────────────────────────────────────────────
# Canonical model inputs : distance_to_goal, shot_angle_rad,
#                          is_penalty, is_direct_free_kick,
#                          is_left_foot, is_right_foot, is_header
# Target                 : is_goal
# x, y                   : retained for auditability — not model inputs
# shot_angle_deg         : inspection/reporting only — not a model input
# shot_body_part         : inspection only — do NOT encode alongside booleans
# Provenance (not features): winner, match_status, match_duration,
#   match_date_local_raw, roundId, seasonId, competitionId, match_label
# ───────────────────────────────────────────────────────────────────────────────
SAVE_COLS = [
    # Shot identity / event
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod", "eventName", "subEventName", "x", "y", "is_goal",
    # Attempt-type flags (upstream, not re-derived)
    "is_penalty", "is_direct_free_kick",
    # Match metadata (provenance — not model inputs)
    "match_datetime_utc", "match_date", "match_date_local_raw",
    "gameweek", "roundId", "seasonId", "competitionId",
    "match_label", "match_status", "match_duration", "winner",
    # Chronological ordering
    "period_sort_key", "seconds_from_match_start",
    "shot_sequence_in_match", "shot_sequence_team_in_match",
    # Pre-shot features — safe for xG model use
    "is_left_foot", "is_right_foot", "is_header", "shot_body_part",
    "distance_to_goal", "shot_angle_rad", "shot_angle_deg",
]

train = shots[shots["league"] != "England"][SAVE_COLS].copy()
test  = shots[shots["league"] == "England"][SAVE_COLS].copy()

assert set(train["league"].unique()) == {"France","Germany","Italy","Spain"}, "Wrong leagues in train"
assert set(test["league"].unique())  == {"England"}, "Wrong leagues in test"

shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_xg_baseline.parquet",       index=False)
train.to_parquet(DATA_DIR / "wyscout_train_xg_baseline.parquet",            index=False)
test.to_parquet(DATA_DIR / "wyscout_test_xg_baseline.parquet",              index=False)
shots[SAVE_COLS].head(100).to_csv(DATA_DIR / "wyscout_xg_baseline_sample.csv", index=False)

print(f"Saved {len(shots):,} combined → wyscout_xg_baseline.parquet")
print(f"Train: {len(train):,}  |  Test: {len(test):,}")

Saved 43,040 combined → wyscout_xg_baseline.parquet
Train: 34,159  |  Test: 8,881


## Step 10: Reload verification

In [12]:
check = pd.read_parquet(DATA_DIR / "wyscout_xg_baseline.parquet")
assert check.shape == shots[SAVE_COLS].shape, "Shape mismatch on reload"
assert set(SAVE_COLS) == set(check.columns), "Column mismatch on reload"

# Confirm locked model inputs are null-free in all three splits
LOCKED_INPUTS = [
    "distance_to_goal", "shot_angle_rad",
    "is_penalty", "is_direct_free_kick",
    "is_left_foot", "is_right_foot", "is_header",
    "is_goal",
]
train_check = pd.read_parquet(DATA_DIR / "wyscout_train_xg_baseline.parquet")
test_check  = pd.read_parquet(DATA_DIR / "wyscout_test_xg_baseline.parquet")
for col in LOCKED_INPUTS:
    assert check[col].notna().all(),       f"Null in combined {col}"
    assert train_check[col].notna().all(), f"Null in train {col}"
    assert test_check[col].notna().all(),  f"Null in test {col}"

# Dtype checks
assert pd.api.types.is_float_dtype(check["distance_to_goal"]), "distance_to_goal not float"
assert pd.api.types.is_float_dtype(check["shot_angle_rad"]),   "shot_angle_rad not float"
for bool_col in ["is_left_foot","is_right_foot","is_header","is_penalty","is_direct_free_kick"]:
    assert check[bool_col].isin([0, 1, True, False]).all(), f"{bool_col} not binary"
assert check["is_goal"].isin([0, 1]).all(), "is_goal not binary"

print(check.shape)
check.head()

(43040, 35)


,league,matchId,playerId,teamId,eventSec,matchPeriod,eventName,subEventName,x,y,...,seconds_from_match_start,shot_sequence_in_match,shot_sequence_team_in_match,is_left_foot,is_right_foot,is_header,shot_body_part,distance_to_goal,shot_angle_rad,shot_angle_deg
0,France,2500691,353833,19830,241.432822,1H,Shot,Shot,91,30,...,241.432822,1,1,False,True,False,right_foot,16.560873,0.259204,14.851312
1,France,2500691,288423,3780,339.735142,1H,Shot,Shot,92,59,...,339.735142,2,1,False,True,False,right_foot,10.392998,0.576270,33.017814
2,France,2500691,3450,19830,576.442938,1H,Shot,Shot,91,54,...,576.442938,3,2,False,False,True,head,9.833662,0.692985,39.705133
3,France,2500691,28922,19830,742.946020,1H,Shot,Shot,82,51,...,742.946020,4,3,False,True,False,right_foot,18.912229,0.382101,21.892781
4,France,2500691,353833,19830,828.687717,1H,Shot,Shot,91,69,...,828.687717,5,4,False,True,False,right_foot,16.007152,0.277512,15.900243
